# Preprocessing - Outer Join labels va slurm log

Muc tieu:
- Outer join 2 bang `labelled_jobids.csv` va `slurm-log.csv` theo `id_job`.
- Label khong co se duoc gan `None`.
- Chi giu lai tap cot ad-hoc theo yeu cau.
- Ghi ket qua ra file moi trong thu muc `data/`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

In [2]:
DATA_DIR = Path('..') / 'data'
LABEL_PATH = DATA_DIR / 'labelled_jobids.csv'
SLURM_PATH = DATA_DIR / 'slurm-log.csv'
OUTPUT_PATH = DATA_DIR / 'jobs.csv'

labels = pd.read_csv(LABEL_PATH, dtype={'id_job': 'string', 'model': 'string'})
slurm = pd.read_csv(SLURM_PATH, dtype={'id_job': 'string'})

print('labels shape:', labels.shape)
print('slurm shape :', slurm.shape)

labels shape: (3430, 2)
slurm shape : (395914, 29)


In [3]:
# Cac cot ad-hoc can giu lai
requested_slurm_cols = [
    'id_job',
    'id_array_job',
    'id_array_task',
    'id_user',
    'cpus_req',
    'mem_req',
    'partition',
    'constraints',
    'timelimit',
    'time_submit',
    'time_eligible',
    'flags',
    'track_steps',
    'job_type',
    'tres_req',
    'array_max_tasks',
    'priority',
]

missing_slurm_cols = [c for c in requested_slurm_cols if c not in slurm.columns]
if missing_slurm_cols:
    print('Missing columns in slurm-log.csv:', missing_slurm_cols)

for c in missing_slurm_cols:
    slurm[c] = pd.NA

slurm_keep = slurm[requested_slurm_cols].copy()

labels_keep = labels[['id_job', 'model']].copy()
labels_keep = labels_keep.drop_duplicates(subset=['id_job'], keep='first')

print('slurm_keep shape :', slurm_keep.shape)
print('labels_keep shape:', labels_keep.shape)

slurm_keep shape : (395914, 17)
labels_keep shape: (3430, 2)


In [4]:
# Outer join theo id_job, label thieu -> None
merged = slurm_keep.merge(labels_keep, on='id_job', how='outer')
merged['model'] = merged['model'].fillna('None')

# Gom model vao 3 nhom: vision, language, graph
def classify_model(model_name: str) -> str:
    if pd.isna(model_name):
        return 'None'
    m = str(model_name).strip().lower()

    # Language models
    if ('bert' in m) or ('distilbert' in m):
        return 'language'

    # Graph neural networks
    if (
        ('dimenet' in m)
        or ('schnet' in m)
        or ('pna' in m)
        or (m == 'conv')
        or ('nnconv' in m)
    ):
        return 'graph'

    # Vision models
    if (
        ('vgg' in m)
        or ('resnet' in m)
        or ('inception' in m)
        or m.startswith('u3-')
        or m.startswith('u4-')
        or m.startswith('u5-')
        or ('u-net' in m)
        or ('unet' in m)
    ):
        return 'vision'

    return 'None'

# Gom model ve base model cu the
def classify_base_model(model_name: str) -> str:
    if pd.isna(model_name):
        return 'None'
    m = str(model_name).strip().lower()

    # Vision base models
    if 'vgg' in m:
        return 'VGG'
    if 'resnet' in m:
        return 'ResNet'
    if 'inception' in m:
        return 'Inception'
    if (
        m.startswith('u3-')
        or m.startswith('u4-')
        or m.startswith('u5-')
        or ('u-net' in m)
        or ('unet' in m)
    ):
        return 'U-Net'

    # Language base models
    if 'distilbert' in m:
        return 'DistillBert'
    if 'bert' in m:
        return 'Bert'

    # Graph base models
    if 'dimenet' in m:
        return 'DimeNet'
    if 'schnet' in m:
        return 'SchNet'
    if 'pna' in m:
        return 'PNA'
    if ('nnconv' in m) or (m == 'conv'):
        return 'NNConv'

    return 'None'

merged['model_group'] = merged['model'].apply(classify_model)
merged['base_model'] = merged['model'].apply(classify_base_model)

# Lam sach sentinel values
mem_req_numeric = pd.to_numeric(merged['mem_req'], errors='coerce')
timelimit_numeric = pd.to_numeric(merged['timelimit'], errors='coerce')
id_array_task_numeric = pd.to_numeric(merged['id_array_task'], errors='coerce')

merged['mem_req_is_sentinel'] = mem_req_numeric >= 9_000_000_000_000_000_000
merged['timelimit_is_sentinel'] = timelimit_numeric == 4294967295
merged['id_array_task_is_sentinel'] = id_array_task_numeric == 4294967294

merged.loc[merged['mem_req_is_sentinel'], 'mem_req'] = pd.NA
merged.loc[merged['timelimit_is_sentinel'], 'timelimit'] = pd.NA
merged.loc[merged['id_array_task_is_sentinel'], 'id_array_task'] = pd.NA

final_cols = requested_slurm_cols + [
    'model',
    'model_group',
    'base_model',
    'mem_req_is_sentinel',
    'timelimit_is_sentinel',
    'id_array_task_is_sentinel',
]
result = merged[final_cols].copy()

print('result shape:', result.shape)
display(result.head(10))

result shape: (395914, 23)


,id_job,id_array_job,id_array_task,id_user,cpus_req,mem_req,partition,constraints,timelimit,time_submit,time_eligible,flags,track_steps,job_type,tres_req,array_max_tasks,priority,model,model_group,base_model,mem_req_is_sentinel,timelimit_is_sentinel,id_array_task_is_sentinel
0,10000305171013,8387626672597,193.0,77492909145526,1,NaN,normal,\N,NaN,1629149775,1629149775,4,0,OTHER,"1=1,2=8500,4=1,5=1,1002=2",0,10125,None,None,None,True,True,False
1,10000639236651,4894706564869,106.0,42832188225543,4,NaN,normal,\N,525600.0,1632424776,1632424777,8,0,LLSUB:BATCH,"1=4,2=34000,4=1,5=4,1002=1",0,10419,None,None,None,True,False,False
2,10000709074588,84173100828392,68.0,11289836447956,1,NaN,normal,xeon-g6,NaN,1609809099,1609809100,4,0,OTHER,"1=1,2=8500,4=1,5=1",0,10771,None,None,None,True,True,False
3,10000736577517,76874831410064,60.0,91080490429722,1,NaN,xeon-p8,\N,4320.0,1621829754,1621829754,4,0,LLSUB:BATCH,"1=1,2=4000,4=1,5=1",0,10235,None,None,None,True,False,False
4,1000102612885,39812615468826,118.0,36880203429568,1,NaN,xeon-p8,\N,525600.0,1620783465,1620783465,8,0,LLSUB:BATCH,"1=1,2=4000,4=1,5=1",0,10152,None,None,None,True,False,False
5,10001446352148,9485740249803,182.0,28791209246369,1,NaN,normal,xeon-g6&6248,NaN,1612461599,1612461600,4,0,LLMAPREDUCE:MAP,"1=1,2=8500,4=1,5=1",0,10128,None,None,None,True,True,False
6,10001693543031,16618712154521,NaN,1706828023724,8,NaN,normal,xeon-g6,7200.0,1613755018,1613755018,8,0,OTHER,"1=8,2=15776,4=1,5=8",0,10205,None,None,None,True,False,True
7,10001859171470,16618712154521,NaN,91613978413847,1,10240.0,normal,xeon-g6,1440.0,1618206106,1618206106,4,0,OTHER,"1=1,2=10240,4=1,5=1",0,10482,None,None,None,False,False,True
8,10001952220349,31110270194034,3061.0,42770088536256,20,NaN,normal,xeon-g6,NaN,1612649635,1612649635,8,0,OTHER,"1=20,2=170000,4=1,5=20",0,10155,None,None,None,True,True,False
9,10002272950475,32716616865493,23465.0,45487591934198,1,NaN,normal,xeon-g6,NaN,1615952800,1615952800,4,0,LLSUB:BATCH,"1=1,2=8500,4=1,5=1",0,10447,None,None,None,True,True,False


In [5]:
# Kiem tra nhanh truoc khi ghi file
stats = {
    'total_rows': len(result),
    'unique_id_job': result['id_job'].nunique(dropna=True),
    'label_none_rows': int((result['model'] == 'None').sum()),
    'label_present_rows': int((result['model'] != 'None').sum()),
    'mem_req_sentinel_rows': int(result['mem_req_is_sentinel'].sum()),
    'timelimit_sentinel_rows': int(result['timelimit_is_sentinel'].sum()),
    'id_array_task_sentinel_rows': int(result['id_array_task_is_sentinel'].sum()),
}

display(pd.DataFrame([stats]))
display(result['model'].value_counts(dropna=False).head(20).rename('count').to_frame())

group_jobcount = (
    result[result['model_group'].isin(['vision', 'language', 'graph'])]
    .groupby('model_group')['id_job']
    .nunique()
    .reindex(['vision', 'language', 'graph'], fill_value=0)
    .rename('job_count')
    .to_frame()
    .reset_index()
    .rename(columns={'model_group': 'group'})
)
display(group_jobcount)

,total_rows,unique_id_job,label_none_rows,label_present_rows,mem_req_sentinel_rows,timelimit_sentinel_rows,id_array_task_sentinel_rows
0,395914,395914,392484,3430,346766,173875,185549


,count
model,
None,392484
inception4,243
inception3,241
vgg19,199
bert-base-uncased,189
vgg11,185
vgg16,176
distilbert-base-uncased,172
U3-32,165


,group,job_count
0,vision,2938
1,language,361
2,graph,131


In [6]:
result.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote file: {OUTPUT_PATH.resolve()}')

Wrote file: C:\Users\Admin\Documents\Tuan_Huy\MOE\data\jobs.csv


In [7]:
# Feature engineering + export 3 files theo yeu cau
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

UNLABELED_OUTPUT_PATH = DATA_DIR / 'jobs_unlabeled_scaled.csv'
TRAIN_OUTPUT_PATH = DATA_DIR / 'jobs_labeled_train_scaled.csv'
TEST_OUTPUT_PATH = DATA_DIR / 'jobs_labeled_test_scaled.csv'

feature_base = result[[
    'id_job', 'model', 'model_group', 'base_model',
    'priority', 'cpus_req', 'mem_req', 'timelimit', 'time_submit', 'time_eligible',
    'partition', 'constraints', 'flags', 'job_type',
]].copy()

# Numeric features
feature_base['priority'] = pd.to_numeric(feature_base['priority'], errors='coerce')
feature_base['cpus_req'] = pd.to_numeric(feature_base['cpus_req'], errors='coerce')
feature_base['mem_req'] = pd.to_numeric(feature_base['mem_req'], errors='coerce')
feature_base['time_limit'] = pd.to_numeric(feature_base['timelimit'], errors='coerce')

feature_base['mem_req'] = feature_base['mem_req'].fillna(feature_base['mem_req'].median())
feature_base['time_limit'] = feature_base['time_limit'].fillna(feature_base['time_limit'].median())

# Time features tu time_submit: hour/dayofweek/month -> sin/cos
ts_submit = pd.to_numeric(feature_base['time_submit'], errors='coerce')
submit_dt = pd.to_datetime(ts_submit, unit='s', errors='coerce')

feature_base['submit_hour'] = submit_dt.dt.hour
feature_base['submit_dayofweek'] = submit_dt.dt.dayofweek
feature_base['submit_month'] = submit_dt.dt.month

feature_base['submit_hour_sin'] = np.sin(2 * np.pi * feature_base['submit_hour'] / 24.0)
feature_base['submit_hour_cos'] = np.cos(2 * np.pi * feature_base['submit_hour'] / 24.0)
feature_base['submit_dayofweek_sin'] = np.sin(2 * np.pi * feature_base['submit_dayofweek'] / 7.0)
feature_base['submit_dayofweek_cos'] = np.cos(2 * np.pi * feature_base['submit_dayofweek'] / 7.0)
feature_base['submit_month_sin'] = np.sin(2 * np.pi * (feature_base['submit_month'] - 1) / 12.0)
feature_base['submit_month_cos'] = np.cos(2 * np.pi * (feature_base['submit_month'] - 1) / 12.0)

# eligible_delay = time_eligible - time_submit
ts_eligible = pd.to_numeric(feature_base['time_eligible'], errors='coerce')
feature_base['eligible_delay'] = ts_eligible - ts_submit

# Category bucketing + one-hot encoding
def top_n_bucket(series: pd.Series, top_n: int) -> pd.Series:
    s = series.fillna('missing').astype(str).str.strip().str.lower()
    top_values = s.value_counts().head(top_n).index
    return s.where(s.isin(top_values), 'other')

partition_bucket = top_n_bucket(feature_base['partition'], 5)
constraint_bucket = top_n_bucket(feature_base['constraints'], 7)
flag_bucket = top_n_bucket(feature_base['flags'], 4)
job_type_bucket = top_n_bucket(feature_base['job_type'], 4)

partition_ohe = pd.get_dummies(partition_bucket, prefix='partition', dtype='int8')
constraint_ohe = pd.get_dummies(constraint_bucket, prefix='constraint', dtype='int8')
flag_ohe = pd.get_dummies(flag_bucket, prefix='flag', dtype='int8')
job_type_ohe = pd.get_dummies(job_type_bucket, prefix='job_type', dtype='int8')

feature_df = pd.concat([
    feature_base[[
        'id_job', 'model', 'model_group', 'base_model',
        'priority', 'cpus_req', 'mem_req', 'time_limit',
        'submit_hour_sin', 'submit_hour_cos',
        'submit_dayofweek_sin', 'submit_dayofweek_cos',
        'submit_month_sin', 'submit_month_cos',
        'eligible_delay',
    ]],
    partition_ohe, constraint_ohe, flag_ohe, job_type_ohe,
], axis=1)

numeric_cols = [
    'priority', 'cpus_req', 'mem_req', 'time_limit',
    'submit_hour_sin', 'submit_hour_cos',
    'submit_dayofweek_sin', 'submit_dayofweek_cos',
    'submit_month_sin', 'submit_month_cos',
    'eligible_delay',
]

feature_df[numeric_cols] = feature_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
feature_df[numeric_cols] = feature_df[numeric_cols].fillna(feature_df[numeric_cols].median(numeric_only=True))

# Tach unlabeled va labeled
unlabeled_mask = feature_df['model'].isna() | (feature_df['model'].astype(str).str.lower() == 'none')
df_unlabeled = feature_df[unlabeled_mask].copy()
df_labeled = feature_df[~unlabeled_mask].copy()

# 1) File unlabeled: fit scaler tren toan bo numeric cua tap unlabeled
scaler_unlabeled = StandardScaler()
if len(df_unlabeled) > 0:
    df_unlabeled[numeric_cols] = scaler_unlabeled.fit_transform(df_unlabeled[numeric_cols])
df_unlabeled.to_csv(UNLABELED_OUTPUT_PATH, index=False)

# 2) + 3) Labeled train/test: stratify theo model
if len(df_labeled) == 0:
    raise ValueError('Khong co record nao co model de split train/test.')

df_train, df_test = train_test_split(
    df_labeled,
    test_size=0.2,
    random_state=42,
    stratify=df_labeled['model']
 )

scaler_labeled = StandardScaler()
df_train[numeric_cols] = scaler_labeled.fit_transform(df_train[numeric_cols])
df_test[numeric_cols] = scaler_labeled.transform(df_test[numeric_cols])

df_train.to_csv(TRAIN_OUTPUT_PATH, index=False)
df_test.to_csv(TEST_OUTPUT_PATH, index=False)

print('unlabeled shape:', df_unlabeled.shape)
print('train shape    :', df_train.shape)
print('test shape     :', df_test.shape)

print('Wrote:', UNLABELED_OUTPUT_PATH.resolve())
print('Wrote:', TRAIN_OUTPUT_PATH.resolve())
print('Wrote:', TEST_OUTPUT_PATH.resolve())

print('\nTrain model distribution:')
display(df_train['model'].value_counts(normalize=True).mul(100).round(2).rename('percent').to_frame())

print('Test model distribution:')
display(df_test['model'].value_counts(normalize=True).mul(100).round(2).rename('percent').to_frame())

unlabeled shape: (392484, 35)
train shape    : (2744, 35)
test shape     : (686, 35)
Wrote: C:\Users\Admin\Documents\Tuan_Huy\MOE\data\jobs_unlabeled_scaled.csv
Wrote: C:\Users\Admin\Documents\Tuan_Huy\MOE\data\jobs_labeled_train_scaled.csv
Wrote: C:\Users\Admin\Documents\Tuan_Huy\MOE\data\jobs_labeled_test_scaled.csv

Train model distribution:


,percent
model,
inception4,7.07
inception3,7.03
vgg19,5.79
bert-base-uncased,5.5
vgg11,5.39
vgg16,5.14
distilbert-base-uncased,5.03
U3-32,4.81
U3-128,4.81


Test model distribution:


,percent
model,
inception4,7.14
inception3,7.0
vgg19,5.83
bert-base-uncased,5.54
vgg11,5.39
vgg16,5.1
distilbert-base-uncased,4.96
U3-32,4.81
U4-32,4.81


## Ghi chu

- Cot `priority` duoc giu nguyen theo du lieu log hien co.
- Neu can dung priority tai thoi diem start, can co snapshot/nguon log theo timeline chi tiet hon.
- Cot moi `model_group` gom 3 nhom: `vision`, `language`, `graph` (ngoai mapping se la `None`).
- Da xu ly sentinel value: `mem_req` (gia tri rat lon gan int64 max), `timelimit = 4294967295`, `id_array_task = 4294967294`.
- Cac cot co `*_is_sentinel` duoc them de truy vet ban ghi da duoc clean.
- Da doi pipeline output thanh 3 file: `data/jobs_unlabeled_scaled.csv`, `data/jobs_labeled_train_scaled.csv`, `data/jobs_labeled_test_scaled.csv`.
- Unlabeled: fit `StandardScaler` tren toan bo numeric cua tap unlabeled.
- Labeled: split stratify theo `model`, fit scaler tren train va transform test.